In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
!pip install ultralytics -q

In [ ]:
# Mount Google Drive (put your dataset there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Change this to where your dataset folder is in Drive
DATASET_PATH = '/content/drive/MyDrive/fire_smoke_dataset'

# Your data.yaml should be inside that folder
DATA_YAML = os.path.join(DATASET_PATH, 'data.yaml')

print('Dataset path:', DATASET_PATH)
print('data.yaml exists:', os.path.exists(DATA_YAML))

In [ ]:
# data.yaml format reminder (edit yours to match):
# path: /content/drive/MyDrive/fire_smoke_dataset
# train: images/train
# val:   images/val
# nc: 2
# names: ['fire', 'smoke']

with open(DATA_YAML, 'r') as f:
    print(f.read())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # n=nano (fast), swap to yolov8s/m/l for better accuracy

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name='fire_smoke_v1',
    project='/content/drive/MyDrive/runs',  # saves to Drive so you don't lose it
    patience=10,   # early stop if no improvement for 10 epochs
    device=0       # GPU
)

In [ ]:
# Show training curves
from IPython.display import Image
Image('/content/drive/MyDrive/runs/fire_smoke_v1/results.png')

In [ ]:
# Validate on val set
best = YOLO('/content/drive/MyDrive/runs/fire_smoke_v1/weights/best.pt')
best.val(data=DATA_YAML, device=0)

In [ ]:
# Test on a single image or video
TEST_SOURCE = '/content/drive/MyDrive/test.jpg'  # change to your file

best.predict(
    source=TEST_SOURCE,
    conf=0.25,
    save=True,
    project='/content/drive/MyDrive/runs',
    name='test_output'
)

In [ ]:
# Show prediction result
import glob
from IPython.display import Image

imgs = glob.glob('/content/drive/MyDrive/runs/test_output/*.jpg')
Image(imgs[0])

In [ ]:
# Export to ONNX (optional, for deployment)
best.export(format='onnx')